# 🚀 Mermaid Diagram Generator – Gemma-2-2B-IT on **Colab TPU v5e** (PyTorch + Torch XLA)

**Optimized for Google Colab TPU v5e (free tier)**
- Native TPU acceleration via **torch_xla** (no Unsloth – Unsloth does not support TPU)
- LoRA rank 16 targeting **all linear modules** (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`)
- **bfloat16** (no 4-bit NF4 – bitsandbytes is CUDA-only)
- Full pipeline: fine-tune → merge → INT8 .tflite via **ai-edge-torch GenAI**
- Custom Mermaid validation with `mmdc` (sandbox bypass)

**Change runtime to TPU v5e** before running.
Training ≈ **45–70 minutes** on TPU v5e (much faster than T4).

Final artifact: `gemma_mermaid_diagram_generator.tflite` ready for LiteRT edge deployment.

In [ ]:
%%capture
# TPU v5e setup
!pip install -q --upgrade transformers==4.46.0 peft==0.13.2 trl==0.11.1 accelerate datasets huggingface_hub
!pip install -q ai-edge-torch-nightly

# Torch XLA for TPU (Colab TPU v5e already has it, but ensure latest)
!pip install -q torch-xla[tpu] -f https://storage.googleapis.com/libtpu-releases/wheels/tpuvm/torch_xla-2.5.0-cp310-cp310-linux_x86_64.whl

# System deps for Mermaid CLI
!apt-get update -qq && apt-get install -y nodejs npm chromium-browser libnss3 libgconf-2-4 libfontconfig1 -qq
!npm install -g @mermaid-js/mermaid-cli@10.9.1 --silent

import torch
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl
import ast
import subprocess
import os
import json
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

# TPU device
device = xm.xla_device()
print(f"✅ Using TPU device: {device}")

In [ ]:
# ===========================
# 2. Data Loading & Gemma Formatting (same as fixed version)
# ===========================
model_id = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset("colinfrisch/diagrams-mermaid-filtered", split="train")
split_dataset = dataset.train_test_split(test_size=20, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

def format_row(example):
    try:
        raw_msg = example["messages"].replace("'role': user", "'role': 'user'").replace("'role': assistant", "'role': 'assistant'")
        messages = ast.literal_eval(raw_msg)
        for msg in messages:
            if msg.get("role") == "assistant":
                msg["role"] = "model"
        formatted_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        return {"text": formatted_text}
    except Exception:
        return {"text": None}

train_dataset = train_dataset.map(format_row, remove_columns=train_dataset.column_names)
train_dataset = train_dataset.filter(lambda x: x["text"] is not None)

print(f"Training dataset size: {len(train_dataset)}")
print(f"Evaluation dataset size (raw): {len(eval_dataset)}")

In [ ]:
# ===========================
# 3. Model + LoRA on TPU (bfloat16 – no 4-bit NF4 on TPU)
# ===========================
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto"  # torch_xla will handle placement
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Move to TPU device
model = model.to(device)
print("✅ Model + LoRA loaded on TPU v5e")

In [ ]:
# ===========================
# 4. Training with SFTTrainer on TPU
# ===========================
training_args = SFTConfig(
    output_dir="./gemma-mermaid-lora-tpu",
    num_train_epochs=3,
    per_device_train_batch_size=4,          # TPU v5e can handle larger batches
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="adamw_8bit",
    logging_steps=10,
    save_strategy="epoch",
    learning_rate=2e-4,
    bf16=True,
    max_seq_length=2048,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    peft_config=lora_config,
    tokenizer=tokenizer,
    args=training_args,
)

trainer.train()
trainer.save_model("gemma-mermaid-lora-final")
print("✅ Training completed on TPU v5e")

In [ ]:
# ===========================
# 5. Custom Mermaid Validation (same robust version)
# ===========================
def validate_mermaid(mmd_code, filename="test.mmd"):
    with open(filename, "w") as f:
        f.write(mmd_code)
    try:
        config_path = "puppeteer-config.json"
        with open(config_path, "w") as f:
            json.dump({"args": ["--no-sandbox", "--disable-setuid-sandbox", "--disable-dev-shm-usage"]}, f)
        result = subprocess.run(
            ["mmdc", "-i", filename, "-o", "test.svg", "-p", config_path],
            capture_output=True, text=True, timeout=30
        )
        return result.returncode == 0, result.stdout + result.stderr
    except Exception as e:
        return False, str(e)

success_count = 0
model.eval()
for idx, example in enumerate(eval_dataset):
    try:
        raw_msg = example["messages"].replace("'role': user", "'role': 'user'").replace("'role': assistant", "'role': 'assistant'")
        messages = ast.literal_eval(raw_msg)
        user_content = messages[0]["content"]

        prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": user_content}],
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        input_length = inputs.input_ids.shape[1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

        generated = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

        mmd = None
        if "```mermaid" in generated:
            start = generated.find("```mermaid") + 11
            end = generated.find("```", start)
            mmd = generated[start:end].strip()

        if mmd:
            valid, _ = validate_mermaid(mmd)
            success_count += int(valid)
            print(f"Sample {idx+1}/20 | Valid: {valid}")
        else:
            print(f"Sample {idx+1}/20 | No Mermaid block found")
    except Exception as e:
        print(f"Sample {idx+1}/20 | Error: {e}")

print(f"\n✅ Mermaid compilation success rate: {success_count}/20 = {success_count/20*100:.1f}%")

In [ ]:
# ===========================
# 6. Merge LoRA → Standalone PyTorch
# ===========================
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="cpu"
)
peft_model = PeftModel.from_pretrained(base_model, "gemma-mermaid-lora-final", device_map="cpu")
merged_model = peft_model.merge_and_unload()

merged_model.save_pretrained("gemma-mermaid-merged")
tokenizer.save_pretrained("gemma-mermaid-merged")
print("✅ LoRA merged into standalone PyTorch model")

In [ ]:
# ===========================
# 7. Conversion to .tflite (INT8) via ai-edge-torch GenAI
# ===========================
import ai_edge_torch.genai.torch_models as edge_models
import ai_edge_torch.genai as genai

model_path = "./gemma-mermaid-merged"

try:
    edge_gemma = edge_models.gemma2.Gemma2(model_path)
    
    quant_config = genai.quantize.QuantConfig(
        weight_quant=genai.quantize.DynamicInt8WeightQuant()
    )
    
    print("🚀 Starting GenAI conversion...")
    converter = genai.Converter(edge_gemma, quant_config)
    tflite_model = converter.convert()
    
    with open("gemma_mermaid_diagram_generator.tflite", "wb") as f:
        f.write(tflite_model)
    print("✅ Successfully converted to gemma_mermaid_diagram_generator.tflite (INT8)")
except Exception as e:
    print(f"❌ Conversion failed: {e}")
    print("Note: Run conversion on a high-RAM machine if needed (TFLite export is CPU-based).")